<!--

    Gaia Data Processing and Analysis Consortium (DPAC) 
    Co-ordination Unit 9 Work Package 930
    
    (c) 2005-2025 Gaia DPAC
    
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
    -->

The science exploitation platform currently hosts the following data sets:

* The full set of Gaia DR3 tables including the main source catalogue (e.g. `gaiadr3.gaia_source`) and all other astrophysical parameter and spectroscopy data presented in the same name space as in the Gaia archive;
* The Gaia EDR3 source catalogue (direct copies of `gaiaedr3.gaia_source` from the Gaia archive) along with Bailer-Jones distances corresponding to `gaiaedr3.gaiaedr3_distance`; and
* Best neighbour tables for PanSTARRS DR1, 2MASS PSC and ALLWISE (again, directly mirrored from the corresponding tables in the Gaia archive under the gaiaedr3 / gaiadr3 namespace) along with the original records from the 2MASS PSC and ALLWISE catalogues, and Object Thin / Object Mean columns from PanSTARRS DR1 (respectively 2MASS, ALLWISE and PS1 hereafter) where a best neighbour is available.

These are presented as queryable resources as demonstrated in the following cells and subsequent notebooks. Each cell can be run by pressing the "Run" icon above; hover over other icons to see more options, e.g. to clear current output. Note that the SQL-like API for querying here is [PySpark SQL](https://spark.apache.org/docs/3.5.2/sql-programming-guide.html), not ADQL.

In [1]:
# Import the spark config
from astroflow_spark_gaia import spark

# show the database(s) available within the context
spark.sql("show databases").show()

# show the contents of the catalogue store that has been set up
print("Tables available in current database context:")
for line in spark.catalog.listTables(): print (line)

# show the first 20 records from the Gaia DR3 source catalogue
print("\nFirst 20 entries in gaia_source:")
spark.sql("select * from gaiadr3.gaia_source").show()

+---------+
|namespace|
+---------+
|  default|
|  gaiadr3|
| gaiaedr3|
+---------+

Tables available in current database context:


Table(name='gaia_source', catalog='spark_catalog', namespace=['gaiaedr3'], description=None, tableType='EXTERNAL', isTemporary=False)
Table(name='gaia_source_allwise_best_neighbours', catalog='spark_catalog', namespace=['gaiaedr3'], description=None, tableType='EXTERNAL', isTemporary=False)
Table(name='gaia_source_ps1_best_neighbours', catalog='spark_catalog', namespace=['gaiaedr3'], description=None, tableType='EXTERNAL', isTemporary=False)
Table(name='gaia_source_tmasspsc_best_neighbours', catalog='spark_catalog', namespace=['gaiaedr3'], description=None, tableType='EXTERNAL', isTemporary=False)

First 20 entries in gaia_source:


+-------------------+--------------------+----------------+------------+---------+------------------+-----------+------------------+-----------+--------------------+--------------+-------------------+---------+-------------------+-----------+--------------------+-----------+------------+----------------+------------+-------------+-----------------+-------------+--------------+------------------+-------------------+---------------+--------------------+--------------------+-------------------------+------------------------+------------------+-------------------+------------------------+----------------------------+-------------------------+------------------------+-------------------------+------------+------------------+--------------------+---------------------+--------------------------+----------------------+-----------------------+----------------------------+-----------------------+-----------------------+----------------+--------------------+------------------------+--------------

The database context is Gaia DR3 (`gaiadr3.`) at start-up so all that is needed to identify a DR3 table is the name (the first item in each record above) in queries. To change from the default context, e.g. to Gaia EDR3, simply

    spark.sql("USE gaiaedr3")
or preface table names with the database name (`gaiaedr3.gaia_source`). Be aware that some tables (notably `gaia_source`) exist with the same name in different versions (i.e. different column and/or row sets) in different databases so be sure to use the correct one for your workflow - it is probably a good idea to always prefix table names with the required database name just to be sure.



Column descriptions for `gaia_source` are browsable within the Gaia archive ADQL search tree-view, and also (for Gaia DR3 for example) [online](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html), as are the [best neighbour table columns](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_cross-matches/). For the "external" catalogues the column sets can be browsed as follows (click on each to follow the links):

* [2MASS PSC](https://irsa.ipac.caltech.edu/data/2MASS/docs/releases/allsky/doc/sec2_2a.html "2MASS point-source catalogue column details");
* [ALLWISE](https://wise2.ipac.caltech.edu/docs/release/allwise/expsup/sec2_1a.html "ALLWISE catalogue column details");
* [PS1 OTMO](https://outerspace.stsci.edu/display/PANSTARRS/PS1+MeanObjectView+table+fields "PS1 DR1 object-thin object-mean catalogue column details") (note that column names follow the Gaia archive convention of all lower-case with underscore separators rather than the original CamelCase).

It is important to note that the best neighbour tables should not be used as a proxy for the original external survey datasets: only those records that are the best neighbour to one or more Gaia sources are included here. If you wish to query and analyse one or other of those catalogues in isolation, this is not the facility to use! The phrase "one or more" is also noteworthy: because of the way the Gaia cross-matches have been derived any one external catalogue source can be the best neighbour to more than one Gaia source. Please consult the [Gaia cross-match documentation on-line](https://gea.esac.esa.int/archive/documentation/GDR3/Catalogue_consolidation/chap_crossmatch/) to familiarise yourself with the details.

Note also the syntax in the last query above. At first sight this might look insanely profligate, especially if you are familiar with existing relational systems serving Gaia data via ADQL/TAP. On this Spark platform the execution engine works rather differently: a "lazy" execution plan is assembled from the workflow as notebook statements and cells are "played" through this interface - nothing actually happens until absolutely necessary. So the select statement is not executed: rather an optimised plan is created from the combination of the query as couched in SQL here, and the ".show()" action. The latter is short-hand for displaying the first 20 entries only, and this is factored into the execution resulting in fast turn-around and a quick look at the underlying data resource. In general, all filters and other optimisations are "pushed down" through the execution plan at the last possible moment (when data are required to be collected from the distributed workers to the master executor in order to action an operation). This is discussed further in other example notebooks.

Column details can also be examined within the notebook by defining a data frame object against any table then echoing the schema; subsequent cells demonstrate some simple queries: 


In [2]:
# define a data frame object on a table:
df = spark.sql("select * from gaiaedr3.gaia_source_ps1_best_neighbours")

# display the schema with a touch of friendly formatting:
print("Column details of gaiaedr3.gaia_source_ps1_best_neighbours:\n")
for field in df.schema: print("%30s : %10s"%(field.name, field.dataType.simpleString()))

Column details of gaiaedr3.gaia_source_ps1_best_neighbours:

                     source_id :     bigint
          clean_panstarrs1_oid :     bigint
        original_ext_source_id :     bigint
              angular_distance :      float
          number_of_neighbours :    tinyint
               number_of_mates :    tinyint
                       xm_flag :   smallint
                        obj_id :     bigint
                 projection_id :   smallint
                   sky_cell_id :   smallint
                 obj_info_flag :        int
                  quality_flag :   smallint
                       ra_mean :     double
                      dec_mean :     double
                   ra_mean_err :      float
                  dec_mean_err :      float
                    epoch_mean :     double
            n_stack_detections :   smallint
                  n_detections :   smallint
                            ng :   smallint
                            nr :   smallint
               

In [3]:
# querying the main catalogue table is as simple as forming an SQL query:
df = spark.sql("select source_id, ra, dec, phot_g_mean_mag AS G from gaiadr3.gaia_source")

df.show()

+----------------+------------------+------------------+---------+
|       source_id|                ra|               dec|        G|
+----------------+------------------+------------------+---------+
|   8250632981888| 44.99858169313919|0.2307647023973472|17.166658|
|  47695612200832|45.569258307075636|0.6120955174162661|20.712004|
| 122939144286208| 45.14042204331775|1.0136424416926233|21.280476|
| 123901216903552| 44.88235054165219| 0.821299053565819| 21.45157|
| 175990579969536| 46.04530207535428|0.9106360832988057|20.696348|
| 437090231873536| 45.09317410201131|1.5103127260808669| 20.39429|
| 684072326827008| 46.61369329277792|2.1324293416844107| 15.72844|
| 828795544830976| 46.92768134111832| 2.649427683241421|17.599916|
| 839412703977216| 47.13668817184588| 2.857572782299921|16.880386|
| 963515783506432|45.759066598862915|2.6518398749664978|21.108837|
| 986953420557696|46.419760518039965|2.4904810676643634|16.064753|
|1039588244212608| 46.58026690834601| 2.920179127597523|19.060

In [4]:
# a simple SQL join on Gaia source_id can be used to pull in external catalogue data with Gaia data via the pre-computed best-neighbour joins:
#query = "SELECT g.source_id, ra, dec, phot_g_mean_mag AS G, g_mean_psf_mag AS g_ps1, r_mean_psf_mag AS r_ps1" + \
    #" FROM gaia_source AS g INNER JOIN gaia_source_ps1_best_neighbours AS p ON g.source_id = p.source_id"

query= "SELECT gaia.source_id, gaia.ra, gaia.dec, gaia.phot_g_mean_mag AS gaia_g, ps1.g_mean_psf_mag AS ps1_g, ps1.r_mean_psf_mag AS ps1_r FROM gaiaedr3.gaia_source AS gaia INNER JOIN gaiaedr3.gaia_source_ps1_best_neighbours AS ps1 ON gaia.source_id = ps1.source_id"

# define a data frame object on this query
df = spark.sql(query)

# echo first 20 records of the results set
df.show()

+----------------+------------------+-------------------+---------+-------+-------+
|       source_id|                ra|                dec|   gaia_g|  ps1_g|  ps1_r|
+----------------+------------------+-------------------+---------+-------+-------+
|   6975027056128|44.976206974203215|0.19597223687938073|21.247803|20.8992|19.7456|
|  69410967253632| 45.33365180640389| 0.8065718615590686|16.937292|18.3138|17.0573|
| 225949639778432| 45.44124688351276| 1.1954836699775477|15.183385|15.8463|15.1432|
| 312643555148800| 44.19331490047449| 1.0141376381509917| 20.36981|20.6292|20.3524|
| 442897027616128| 44.91264312888683|  1.492601852016502|20.322088|20.4575|20.1683|
| 474301831162240| 45.48856296035135| 1.8741327094367917| 20.83472|21.5876|20.8279|
| 503129650290944| 44.82557241329517| 1.7467209989897743|     NULL|22.1183|21.5088|
| 648265184276864| 46.15329115883016| 1.8187255026806104|19.259766|21.1136|19.6957|
| 853633342492160|45.887170370964405| 1.9883446636479867|21.079979|  22.13|2

* although SQL-based, the API is much richer than SQL/ADQL in terms of programmability and flexibility in column types - this will become increasingly important in future data releases.
* the are no ADQL geometric functions to limit spatially selections - the platform is designed primarily for large-scale statistical analyses and does not replicate all features of existing relational systems, especially those needed to delimit row sets to small sub-samples.



